# PlaceMux: Complete ML Pipeline
This notebook implements the end-to-end ML pipeline for PlaceMux's Conversion-Quality Check, fully self-contained. It includes data generation, feature engineering, baseline evaluation, model training with explainability, and the segmented conversion-quality check metric.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, confusion_matrix
import joblib
import os

In [2]:
# 1. Synthetic Data Generation
def generate_skills():
    return ["Python", "SQL", "AWS", "Docker", "Kubernetes", "React", "Node.js", "Java", "C++", "Go"]

def generate_data(num_samples=1000, random_state=42):
    np.random.seed(random_state)
    all_skills = generate_skills()
    
    job_skills_req = []
    job_seniority = []
    
    for _ in range(num_samples):
        req = np.random.choice(all_skills, size=np.random.randint(2, 6), replace=False)
        job_skills_req.append(list(req))
        job_seniority.append(np.random.randint(1, 6))
        
    student_skills = []
    student_years_exp = []
    payment_status = []
    payment_options = ["paid", "failed", "pending", "refunded"]
    
    for _ in range(num_samples):
        has = np.random.choice(all_skills, size=np.random.randint(1, 7), replace=False)
        student_skills.append(list(has))
        student_years_exp.append(np.random.randint(0, 10))
        payment_status.append(np.random.choice(payment_options, p=[0.6, 0.15, 0.15, 0.1]))

    df = pd.DataFrame({
        'job_skills': job_skills_req,
        'job_seniority': job_seniority,
        'student_skills': student_skills,
        'student_years_exp': student_years_exp,
        'payment_status': payment_status
    })

    overlap_ratios = []
    labels = []
    for i, row in df.iterrows():
        overlap = len(set(row['student_skills']).intersection(set(row['job_skills'])))
        total_req = len(row['job_skills'])
        ratio = overlap / total_req if total_req > 0 else 0
        overlap_ratios.append(ratio)
        
        is_good = 1 if (ratio >= 0.6 and row['student_years_exp'] >= row['job_seniority']) else 0
        if np.random.rand() < 0.1:
            is_good = 1 - is_good
        labels.append(is_good)
        
    df['overlap_ratio'] = overlap_ratios
    df['is_good_match'] = labels
    return df

def get_train_val_test_splits(df, random_state=42):
    train_df, temp_df = train_test_split(df, test_size=0.4, random_state=random_state)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=random_state)
    return train_df, val_df, test_df

# Generate dataset
df = generate_data(num_samples=1500)
train_df, val_df, test_df = get_train_val_test_splits(df)
print(f"Data Splitting -> Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
train_df.head(3)

Data Splitting -> Train: 900, Val: 300, Test: 300


,job_skills,job_seniority,student_skills,student_years_exp,payment_status,overlap_ratio,is_good_match
602,"[Python, Go, Java, C++, AWS]",4,"[Go, C++]",3,paid,0.4,0
772,"[SQL, AWS]",2,"[Docker, React, Node.js, Python, Go]",2,failed,0.0,0
846,"[Docker, Java, React, Go, Python]",1,"[AWS, Docker, Python]",9,paid,0.4,0


In [3]:
# 2. Feature Engineering
def engineer_features(df):
    X = pd.DataFrame(index=df.index)
    X['overlap_ratio'] = df['overlap_ratio']
    
    missing = []
    for i, row in df.iterrows():
        overlap = len(set(row['student_skills']).intersection(set(row['job_skills'])))
        total_req = len(row['job_skills'])
        missing.append(total_req - overlap)
        
    X['missing_skills_count'] = missing
    X['seniority_gap'] = df['student_years_exp'] - df['job_seniority']
    X['years_exp'] = df['student_years_exp']
    X['job_seniority'] = df['job_seniority']
    return X

X_train = engineer_features(train_df)
y_train = train_df['is_good_match']
X_val = engineer_features(val_df)
y_val = val_df['is_good_match']
X_test = engineer_features(test_df)
y_test = test_df['is_good_match']

X_train.head(3)

,overlap_ratio,missing_skills_count,seniority_gap,years_exp,job_seniority
602,0.4,3,-1,3,4
772,0.0,2,0,2,2
846,0.4,3,8,9,1


In [4]:
# 3. Baseline Ranker
class BaselineRanker:
    def __init__(self, threshold=0.5):
        self.threshold = threshold
        
    def predict(self, df):
        return (df['overlap_ratio'] >= self.threshold).astype(int)

def evaluate_metrics(y_true, y_pred, name="Model"):
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    print(f"[{name}] Precision: {precision:.3f}, Recall: {recall:.3f}, FPR: {fpr:.3f}")
    return {'precision': precision, 'recall': recall, 'fpr': fpr}

baseline = BaselineRanker(threshold=0.6)
baseline_preds = baseline.predict(test_df)
baseline_metrics = evaluate_metrics(test_df['is_good_match'], baseline_preds, "Baseline")

[Baseline] Precision: 0.696, Recall: 0.582, FPR: 0.057


In [5]:
# 4. ML Model definition
class MatchModel:
    def __init__(self):
        self.model = LogisticRegression(random_state=42, max_iter=1000)
        self.feature_names = None
        
    def train(self, X_train, y_train):
        self.model.fit(X_train, y_train)
        self.feature_names = X_train.columns.tolist()
        
    def predict(self, X):
        return self.model.predict(X)
        
    def explain(self, row, features):
        prob = self.model.predict_proba(features)[0, 1]
        decision = "Good Match" if prob >= 0.5 else "Poor Match"
        coefs = self.model.coef_[0]
        contributions = coefs * features.iloc[0].values
        
        feat_contrib = list(zip(self.feature_names, contributions))
        feat_contrib.sort(key=lambda x: x[1], reverse=True)
        
        pos_reasons = [feat for feat, val in feat_contrib if val > 0.2]
        neg_reasons = [feat for feat, val in feat_contrib if val < -0.2]
                
        req_skills = set(row['job_skills'].iloc[0]) if isinstance(row['job_skills'], pd.Series) else set(row['job_skills'])
        student_skills = set(row['student_skills'].iloc[0]) if isinstance(row['student_skills'], pd.Series) else set(row['student_skills'])
        matched = req_skills.intersection(student_skills)
        missing = req_skills - student_skills
        
        explanation = f"Score: {prob:.2f} ({decision}). "
        explanation += f"Matched on {len(matched)}/{len(req_skills)} required skills. "
        if missing:
            explanation += f"Missing: {', '.join(missing)}. "
            
        if pos_reasons:
            explanation += f"Weighted {', '.join(pos_reasons)} positively. "
        if neg_reasons:
            explanation += f"Penalized by {', '.join(neg_reasons)}."
            
        return explanation

# Train and Evaluate
model = MatchModel()
model.train(X_train, y_train)

print("Validation Set:")
val_preds = model.predict(X_val)
evaluate_metrics(y_val, val_preds, "Validation ML")

print("\nTest Set:")
test_preds = model.predict(X_test)
model_metrics = evaluate_metrics(y_test, test_preds, "Test ML")
print(f"\nDelta vs Baseline Precision: {model_metrics['precision'] - baseline_metrics['precision']:.3f}")

Validation Set:
[Validation ML] Precision: 0.556, Recall: 0.169, FPR: 0.033

Test Set:
[Test ML] Precision: 0.737, Recall: 0.255, FPR: 0.020

Delta vs Baseline Precision: 0.041


In [6]:
# Explainability Example
sample_idx = 0
sample_row = test_df.iloc[[sample_idx]]
sample_X = X_test.iloc[[sample_idx]]

explanation = model.explain(sample_row, sample_X)
print(f"Sample Student Payment Status: {sample_row.iloc[0]['payment_status']}")
print(f"Explanation: {explanation}")

Sample Student Payment Status: pending
Explanation: Score: 0.08 (Poor Match). Matched on 1/4 required skills. Missing: Python, Kubernetes, React. Weighted overlap_ratio positively. Penalized by seniority_gap, missing_skills_count.


In [7]:
# 5. Conversion Quality Check (Segmented Metrics)
print("--- Conversion-Quality Check ---")
test_df_copy = test_df.copy()
test_df_copy['predicted'] = model.predict(X_test)

segments = test_df_copy['payment_status'].unique()
for seg in segments:
    seg_df = test_df_copy[test_df_copy['payment_status'] == seg]
    m = evaluate_metrics(seg_df['is_good_match'], seg_df['predicted'], f"Segment [{seg}]")
    
paid_df = test_df_copy[test_df_copy['payment_status'] == 'paid']
unpaid_df = test_df_copy[test_df_copy['payment_status'].isin(['failed', 'pending'])]

print("\n[Skew Check] Paid vs Unpaid (Failed/Pending) Relevance:")
evaluate_metrics(paid_df['is_good_match'], paid_df['predicted'], "Paid")
evaluate_metrics(unpaid_df['is_good_match'], unpaid_df['predicted'], "Unpaid")

--- Conversion-Quality Check ---
[Segment [pending]] Precision: 0.857, Recall: 0.600, FPR: 0.033
[Segment [refunded]] Precision: 0.500, Recall: 0.143, FPR: 0.034
[Segment [paid]] Precision: 0.714, Recall: 0.167, FPR: 0.014
[Segment [failed]] Precision: 0.667, Recall: 0.250, FPR: 0.022

[Skew Check] Paid vs Unpaid (Failed/Pending) Relevance:
[Paid] Precision: 0.714, Recall: 0.167, FPR: 0.014
[Unpaid] Precision: 0.800, Recall: 0.444, FPR: 0.026


{'precision': 0.8,
 'recall': 0.4444444444444444,
 'fpr': np.float64(0.02631578947368421)}